# Business Moat Scoring - Excel to Delta Lake

> **⚠️ Context Notice**: This notebook is part of the **agent-prompts/Businessmoat-Scoring** framework and works entirely within the **Google Ecosystem**. The agent is powered by **Gemini Gems** and generates formatted output that users can directly copy/paste into Google Sheets for consolidation and export to Databricks.

## Purpose
This notebook implements an ETL pipeline with SCD Type 2 historization that:
- Loads Moat Scoring data from an Excel file stored in Unity Catalog Volume
- Transforms column names to Delta Lake-compatible format
- Implements SCD Type 2 to track all changes over time
- Loads data into Unity Catalog Delta table for visualization and trend analysis

## End-to-End Workflow

**1. AI Agent Generation** (agent-prompts/Businessmoat-Scoring with Gemini Gems)
   * Businessmoat Scoring Agent powered by Gemini Gems analyzes companies
   * Agent generates structured, formatted output (tabular format)
   * Output includes 6-dimensional moat assessments (S1-S6)
   * AI Platform: Gemini Gems | Output: Formatted text for direct paste

**2. Manual Data Collection** (Google Ecosystem)
   * User copies formatted output from Gemini Gem (Ctrl+C)
   * User pastes directly into Google Sheets (Ctrl+V)
   * Google Sheets serves as staging area for data consolidation and editing
   * User exports Google Sheets as Excel file (.xlsx)
   * User manually uploads Excel file to Databricks Unity Catalog Volume

**3. ETL Processing** (this notebook)
   * Load Excel from `/Volumes/thekitchen/local/businessmoatscoring/Moat Scoring.xlsx`
   * Transform column names for Delta Lake compatibility
   * Persist to `thekitchen.s.moat_scoring` Delta table

**4. Analysis & Visualization** (downstream)
   * Delta table ready for SQL queries, dashboards, and further analysis
   * Structured data enables trend tracking and portfolio insights

## Architecture
- **Source**: Excel file `/Volumes/thekitchen/local/businessmoatscoring/Moat Scoring.xlsx`
- **Target**: Delta table `thekitchen.s.moat_scoring`
- **Pattern**: SCD Type 2 (Slowly Changing Dimension) with full history tracking
- **Business Key**: Date + Company (identifies unique assessments)
- **Compute**: Databricks Serverless (Python + PySpark + Delta Lake MERGE)

## Technical Workflow
1. Install Excel library (openpyxl)
2. Load Excel file via pandas
3. Convert to Spark DataFrame
4. Transform column names (remove special characters)
5. Convert Date column to DATE type
6. Add SCD Type 2 columns (valid_from DATE, valid_to DATE, is_current INT)
7. Create data hash for change detection
8. MERGE operation (Partial Refresh Pattern):
   - Close out changed records (set valid_to, is_current=0)
   - Insert new and changed records with current date
   - Preserve companies not in Excel (remain current)
9. Track statistics (total/current/historical records)

**Partial Refresh Logic**: Excel contains companies to update/add. Companies NOT in Excel are preserved as current records.

## Data Schema
- **Business columns**: 
  - `Date` (DATE): Assessment date when moat was scored
  - `Company` (STRING): Company name (business key)
  - System Role, 6 Scoring dimensions, Trends, Classifications
- **SCD Type 2 columns**: 
  - `valid_from` (DATE): When record became active in Delta table
  - `valid_to` (DATE): When superseded (NULL for current)
  - `is_current` (INT): Flag indicating active records (1=current, 0=historical)
  - `data_hash` (STRING): SHA-256 hash for change detection
- **Scoring Framework**: S1-S6 metrics measuring competitive moat strength
- **History Tracking**: Complete audit trail of all scoring changes over time

In [0]:
# ==============================================================================
# STEP 1: Install Excel Processing Library
# ==============================================================================
# Install openpyxl to enable pandas Excel file reading
#
# REQUIREMENT:
#   pandas.read_excel() requires openpyxl for .xlsx format
#
# LIBRARY:
#   openpyxl - Excel 2010+ file format support (.xlsx, .xlsm)
#
# NOTE:
#   Kernel restart handled automatically after installation

%pip install openpyxl

In [0]:
# ==============================================================================
# STEP 2: Load Excel File from Unity Catalog Volume
# ==============================================================================
# Load moat scoring data from Excel into Spark DataFrame
#
# SOURCE LOCATION:
#   /Volumes/thekitchen/local/businessmoatscoring/Moat Scoring.xlsx
#
# DATA FLOW:
#   Gemini Gem → Formatted Output → Copy/Paste → Google Sheets → Excel Export → UC Volume → This Notebook
#
# PROCESSING STEPS:
#   1. pandas reads Excel file (handles .xlsx format)
#   2. Convert to Spark DataFrame (enables distributed processing)
#
# OUTPUT:
#   df_spark - Ready for transformation and SCD Type 2 logic

import pandas as pd

# Load Excel file from Volume using pandas
excel_path = "/Volumes/thekitchen/local/businessmoatscoring/Moat Scoring.xlsx"
df_pandas = pd.read_excel(excel_path)

# Convert pandas DataFrame to Spark DataFrame for distributed processing
df_spark = spark.createDataFrame(df_pandas)

# Display DataFrame structure and preview
print(f"Anzahl Zeilen: {df_spark.count()}")
print(f"Anzahl Spalten: {len(df_spark.columns)}")
print(f"\nSchema:")
df_spark.printSchema()
print(f"\nErste Zeilen:")
display(df_spark.limit(5))

---

## STEP 3: Transform & Persist with SCD Type 2

**Goal**: Transform Excel data and load into Unity Catalog with complete history tracking.

**SCD Type 2 Pattern**: Slowly Changing Dimension Type 2 maintains all historical versions:
* **Business Key**: `Company` (unique identifier per company)
* **Assessment Date**: `Date` (DATE) - when moat was assessed, data field not key
* **Valid Period**: `valid_from` / `valid_to` (DATE) - when version was active in Delta
* **Current Flag**: `is_current = 1` latest version, `= 0` historical versions
* **Change Detection**: SHA-256 hash of Date + all 13 data columns

**Process Flow** (6 cells):
1. **Setup** (3A): Import libraries, define column mappings, target table
2. **Transform** (3B): Rename columns, cast Date to DATE type, add SCD columns, create change hash
3. **Validate** (3C): Check table exists, deduplicate source data
4. **Step 1** (3D): Close changed records (MERGE - set valid_to, is_current=0)
5. **Step 2** (3E): Insert new/changed records (APPEND with is_current=1)
6. **Verify** (3F): Display statistics and validate final state

**Update Pattern** (Partial Refresh):
* **Excel** = Companies to update/add (not necessarily all companies)
* **Monthly reassessment**: New Date + updated scores for included companies
* **Change handling**:
  - **New**: Company not in table → INSERT (is_current=1)
  - **Changed**: Company exists, hash differs → CLOSE old + INSERT new
  - **Unchanged**: Company exists, hash identical → NO ACTION
  - **Not in Excel**: Company in table but not in Excel → PRESERVED (stays current)
* **Automatic versioning**: Date change alone triggers new version (Date in hash)
* **Benefit**: Update subset of companies without affecting others

**Benefits**:
* 📊 Track moat score evolution over time
* 🔍 Query historical state at any date
* ✅ Complete audit trail
* 🎯 Efficient hash-based change detection

In [0]:
# ==============================================================================
# STEP 3A: Setup - Imports and Configuration
# ==============================================================================
# Import required libraries and define configuration for SCD Type 2 pipeline
#
# SCD TYPE 2 TRACKING MODEL:
#   Business Key:  Company (unique identifier)
#   Date:          Assessment date (DATE type, data field not key)
#   valid_from:    Delta table record start date (DATE type)
#   valid_to:      Delta table record end date (DATE type, NULL = current)
#   is_current:    1 = latest version, 0 = historical version
#   data_hash:     SHA-256 hash for change detection
#
# CHANGE DETECTION:
#   Hash includes: Date + all 14 data columns (scores, classifications, etc.)
#   Change trigger: ANY field differs → close old + insert new
#
# UPDATE PATTERN:
#   Partial refresh - Excel contains subset of companies to update
#   Companies in Excel: Check for changes, insert/update as needed
#   Companies NOT in Excel: Preserved (remain current)

from pyspark.sql.functions import col, current_timestamp, lit, sha2, concat_ws, coalesce
from delta.tables import DeltaTable

# Column name mapping: Translate German → English, remove special characters
# WHY: Delta Lake columns cannot contain spaces or special chars ( ,;{}()\n\t=:&-)
column_mapping = {
    # German base columns → English
    "Datum": "Date",
    "Unternehmen": "Company",
    "System-Rolle": "System_Role",
    "Klassifizierung": "Classification",
    # Scoring columns with special characters
    "S1: Substitution Test": "S1_Substitution_Test",
    "S2: Source of Enforcement": "S2_Source_of_Enforcement",
    "S3: Replacement Time": "S3_Replacement_Time",
    "S4: Capital & KnowHow": "S4_Capital_KnowHow",
    "S5: Demand Enforcement": "S5_Demand_Enforcement",
    "S6: Moat Decay Test": "S6_Moat_Decay_Test",
    "Moat Vector Trend": "Moat_Vector_Trend",
    "Aggregated Score": "Aggregated_Score",
    "Decay-Adjusted Score": "Decay_Adjusted_Score",
    "Strategisches Takeaway": "Strategisches_Takeaway"
}

# Target table configuration
table_name = "thekitchen.s.moat_scoring"

print(f"✓ Configuration loaded")
print(f"  Target table: {table_name}")
print(f"  Column mappings: {len(column_mapping)} defined")

In [0]:
# ==============================================================================
# STEP 3B: Transform - Add SCD Type 2 Columns & Hash
# ==============================================================================
# Prepare DataFrame with SCD Type 2 tracking fields and change detection hash
#
# FOUR TRANSFORMATIONS:
#   1. Column Renaming:  German/special chars → English/underscores
#   2. Date Type Cast:   Convert Date column from timestamp to DATE
#   3. SCD Metadata:     Add valid_from (DATE), valid_to (DATE), is_current (INT)
#   4. Change Hash:      SHA-256 of Date + all data columns
#
# BUSINESS KEY:
#   Company only (Date is data, not key)
#   Monthly reassessment: new Date → triggers new version
#
# HASH COMPOSITION:
#   Includes: Date + 13 data columns (System_Role, S1-S6, scores, classification, etc.)
#   Excludes: Company (business key, not part of change detection)
#   NULL handling: COALESCE to empty string for consistent hashing

# Step 1: Rename columns to Delta Lake compatible format
df_clean = df_spark
for old_name, new_name in column_mapping.items():
    df_clean = df_clean.withColumnRenamed(old_name, new_name)

# Convert Date column from timestamp to date
from pyspark.sql.functions import to_date
df_clean = df_clean.withColumn("Date", to_date(col("Date")))

print("✓ Column names transformed (German/special chars → English/underscores)")
print(f"  Sample: {', '.join(df_clean.columns[:4])}... ({len(df_clean.columns)} total)")
print("✓ Date column converted to DATE type (timestamp → date)")

# Step 2: Add SCD Type 2 tracking columns
from pyspark.sql.functions import current_date
df_with_scd = df_clean \
    .withColumn("valid_from", current_date()) \
    .withColumn("valid_to", lit(None).cast("date")) \
    .withColumn("is_current", lit(1).cast("int"))

print("✓ SCD Type 2 metadata added")
print("  valid_from: current_date() (DATE type)")
print("  valid_to: NULL (DATE type, marks current records)")
print("  is_current: 1 (active version)")

# Step 3: Create change detection hash
# Hash includes ALL data columns (Date + scores + classifications)
# Excludes only the business key (Company)
# NULL-safe: COALESCE converts NULLs to empty strings
data_columns = [c for c in df_clean.columns if c != 'Company']

df_with_scd = df_with_scd.withColumn(
    "data_hash",
    sha2(
        concat_ws("|", *[coalesce(col(c).cast("string"), lit("")) for c in data_columns]),
        256
    )
)

print(f"✓ Change detection hash created (SHA-256)")
print(f"  Total DataFrame columns: {len(df_with_scd.columns)}")
print(f"  Data columns in hash: {len(data_columns)}")
print(f"  Business key (excluded): Company")
print(f"\n  Hash includes (triggers versioning if ANY changes):")
for i, col_name in enumerate(data_columns, 1):
    marker = " ← Assessment date" if col_name == "Date" else ""
    print(f"    {i:2}. {col_name}{marker}")
print(f"\nℹ️  Monthly reassessment: New Date → hash changes → new version")

In [0]:
# ==============================================================================
# STEP 3C: Validate Table & Prepare Source Data
# ==============================================================================
# Check if target table exists and has proper SCD Type 2 schema
# Deduplicate source data to prevent MERGE conflicts
#
# TABLE VALIDATION:
#   - Check if table exists
#   - Verify SCD Type 2 columns (is_current, valid_from) are present
#   - Drop and recreate if schema is outdated
#
# SOURCE DEDUPLICATION:
#   - If Excel contains duplicate companies, keep only most recent (by Date)
#   - Prevents: DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE error
#   - Business rule: Latest assessment per company wins

# Check if table exists and has SCD Type 2 structure
try:
    delta_table = DeltaTable.forName(spark, table_name)
    existing_schema = spark.table(table_name).columns
    has_scd_columns = 'is_current' in existing_schema and 'valid_from' in existing_schema
    
    if not has_scd_columns:
        print(f"⚠️  Table exists but lacks SCD Type 2 columns - recreating")
        spark.sql(f"DROP TABLE {table_name}")
        table_exists = False
    else:
        table_exists = True
        print(f"✓ Table {table_name} exists with SCD Type 2 structure")
except:
    table_exists = False
    print(f"ℹ️  Table {table_name} does not exist - will create with initial load")

# Deduplicate source data if Excel contains duplicate companies
if table_exists:
    # Read existing current records for change detection
    existing_current = spark.table(table_name).filter("is_current = 1")
    print(f"  Current records in table: {existing_current.count()}")
    
    # Deduplicate: keep only most recent row per company (by Date)
    from pyspark.sql import Window
    from pyspark.sql.functions import row_number
    
    source_count_before = df_with_scd.count()
    window_spec = Window.partitionBy("Company").orderBy(col("Date").desc())
    df_with_scd_deduped = df_with_scd.withColumn("row_num", row_number().over(window_spec)) \
        .filter(col("row_num") == 1) \
        .drop("row_num")
    source_count_after = df_with_scd_deduped.count()
    
    if source_count_before > source_count_after:
        print(f"  ⚠️  Removed {source_count_before - source_count_after} duplicate companies")
        print(f"      Kept most recent assessment per company (by Date)")
    
    # Use deduplicated DataFrame
    df_with_scd_clean = df_with_scd_deduped
    print(f"  Source data ready: {source_count_after} unique companies")
else:
    df_with_scd_clean = df_with_scd
    print(f"  Source data ready: {df_with_scd.count()} companies for initial load")

In [0]:
# ==============================================================================
# STEP 3D: SCD Type 2 - Close Changed Records
# ==============================================================================
# Identify companies whose data changed and mark their old versions as historical
#
# LOGIC:
#   - Match on Business Key: Company only (not Date)
#   - Compare data_hash to detect changes
#   - If hash differs: Set valid_to = now, is_current = 0
#
# CHANGE TRIGGERS (data_hash includes all of these):
#   - Date changed (monthly reassessment)
#   - Any score changed (S1-S6, Aggregated, Decay-Adjusted)
#   - Classification changed
#   - System Role changed
#   - Moat Vector Trend changed
#
# PARTIAL REFRESH BEHAVIOR:
#   - Companies in Excel: Check for changes
#   - Companies NOT in Excel: No action (stay current)

if table_exists:
    print("\n" + "="*60)
    print("SCD TYPE 2 MERGE: Step 1 - Close Changed Records")
    print("="*60)
    
    # MERGE: Close out records where data_hash changed
    delta_table.alias("target").merge(
        df_with_scd_clean.alias("source"),
        "target.Company = source.Company AND target.is_current = 1"
    ).whenMatchedUpdate(
        condition="target.data_hash != source.data_hash",
        set={
            "valid_to": "source.valid_from",
            "is_current": "0"
        }
    ).execute()
    
    print("✓ Closed changed records (set valid_to and is_current=0)")
    print("  Match: Company (business key only)")
    print("  Condition: data_hash differs (any field changed)")
else:
    print("\nSkipping Step 1: Initial load (no existing records to close)")

In [0]:
# ==============================================================================
# STEP 3E: SCD Type 2 - Insert New & Changed Records
# ==============================================================================
# Insert new versions for companies that are new or have changed data
#
# INSERTION LOGIC:
#   Two types of records to insert:
#   1. NEW companies:     Company not in target table at all
#   2. CHANGED companies: Company exists but data_hash differs
#
# DETECTION METHOD:
#   - NEW:     left_anti join (source not in existing_current)
#   - CHANGED: inner join where data_hash differs
#
# RESULT:
#   All inserted records have:
#   - valid_from = current_timestamp
#   - valid_to = NULL
#   - is_current = 1

if table_exists:
    print("\n" + "="*60)
    print("SCD TYPE 2 MERGE: Step 2 - Insert New & Changed Records")
    print("="*60)
    
    # Find NEW companies (not in target)
    df_new = df_with_scd_clean.alias("source").join(
        existing_current.alias("target"),
        col("source.Company") == col("target.Company"),
        "left_anti"
    ).select("source.*")
    
    # Find CHANGED companies (in target but data_hash differs)
    df_changed = df_with_scd_clean.alias("source").join(
        existing_current.alias("target"),
        (col("source.Company") == col("target.Company")) &
        (col("source.data_hash") != col("target.data_hash")),
        "inner"
    ).select("source.*")
    
    # Combine and insert
    df_to_insert = df_new.union(df_changed)
    insert_count = df_to_insert.count()
    
    if insert_count > 0:
        df_to_insert.write.mode("append").saveAsTable(table_name)
        print(f"✓ Inserted {insert_count} records:")
        print(f"  - New companies: {df_new.count()}")
        print(f"  - Changed companies: {df_changed.count()}")
    else:
        print("✓ No changes detected - all companies unchanged")
    
    print("\nℹ️  Companies NOT in Excel: Preserved as current (no action)")
    
else:
    # Initial load: Create table with all data
    print("\n" + "="*60)
    print("INITIAL LOAD: Creating Table")
    print("="*60)
    
    df_with_scd_clean.write.mode("overwrite").saveAsTable(table_name)
    print(f"✓ Created table {table_name}")
    print(f"  Records loaded: {df_with_scd_clean.count()}")
    print("  All records: is_current=1, valid_to=NULL")

In [0]:
# ==============================================================================
# STEP 3F: Verify - Validate SCD Type 2 Implementation
# ==============================================================================
# Display table statistics and validate SCD Type 2 structure
#
# VALIDATION CHECKS:
#   - Total records:      Current + historical versions
#   - Current records:    is_current = 1 (latest per Company)
#   - Historical records: is_current = 0 (superseded versions)
#
# EXPECTED BEHAVIOR:
#   Run 1: historical_count = 0 (initial load, all new)
#   Run 2: historical_count ≈ # changed companies (reassessments)
#   Run N: historical_count grows with each update cycle
#
# OUTPUT:
#   Statistics, schema, sample data

# Read final table and calculate statistics
final_table = spark.table(table_name)
total_count = final_table.count()
current_count = final_table.filter('is_current = 1').count()
historical_count = final_table.filter('is_current = 0').count()

# Display verification report
print("\n" + "="*60)
print(f"TABLE VERIFICATION: {table_name}")
print("="*60)
print(f"\n✓ Record Counts:")
print(f"  Total:       {total_count:>6} (current + historical)")
print(f"  Current:     {current_count:>6} (is_current = 1)")
print(f"  Historical:  {historical_count:>6} (is_current = 0)")

# Interpretation
if historical_count > 0:
    print(f"\n✓ SCD Type 2 is tracking history")
    print(f"  {historical_count} superseded versions preserved")
else:
    print(f"\nℹ️  Initial load - no historical versions yet")

# Display schema
print(f"\n✓ Schema:")
final_table.printSchema()

# Show sample of current records
print(f"\n✓ Sample (5 current records):")
display(final_table.filter('is_current = 1').limit(5))

---

## Context: agent-prompts/Businessmoat-Scoring

**This notebook is an integral component of the Businessmoat-Scoring AI framework:**

* **Data Layer**: Provides structured moat scoring data in `thekitchen.s.moat_scoring`
* **Workflow Position**: ETL layer between AI-generated output (agent-prompts/Businessmoat-Scoring) and human analysis
* **Purpose**: Transforms agent output from Excel into queryable Delta format for dashboards and trend tracking
* **Scoring Framework**: The 6-dimensional structure (S1-S6) is generated by agent-prompts/Businessmoat-Scoring

## Technical Highlights

### Design Patterns
* **SCD Type 2 Historization**: Full audit trail with validity periods and current flags
* **Change Detection**: SHA-256 hash comparison for efficient change identification
* **Delta Lake MERGE**: Native support for upsert operations with ACID guarantees
* **Excel Integration**: Direct Volume access without manual uploads
* **Column Name Sanitization**: Automated transformation for Delta Lake compatibility
* **Two-Stage Processing**: Pandas for Excel → Spark for distributed operations
* **Unity Catalog Native**: Fully integrated with UC schema `thekitchen.s`

### Data Quality
* **71 Companies**: Comprehensive moat scoring across multiple industries
* **6-Dimensional Framework**: S1-S6 metrics covering substitution, enforcement, timing, capital, demand, decay
* **Temporal Data**: Timestamped assessments for trend analysis
* **Classification System**: Core-eligible vs. satellite investment candidates

### Use Cases
* **Historical Trend Analysis**: Track how moat scores evolve over time
* **Point-in-Time Queries**: Query scoring state at any historical date
* **Change Auditing**: Identify when and how assessments changed
* **Strategic Portfolio Construction**: Build portfolios based on moat trajectory
* **Competitive Moat Benchmarking**: Compare current vs. historical strength
* **System Role Identification**: Track evolution of Global Choke Points and System Backbones
* **Decay Tracking**: Monitor moat degradation patterns

### Future Enhancements
* Add data validation checks (score ranges, mandatory fields)
* Create time-series analysis views leveraging SCD history
* Build dashboards showing moat score evolution over time
* Automate via Databricks Jobs for scheduled refreshes
* Implement alerting for significant scoring changes

In [0]:
# ==============================================================================
# Add SystemRoleID Column - Extract Numeric Value from System_Role
# ==============================================================================
# Add SystemRoleID as INT column and populate with numeric value from System_Role
# PRESERVES SCD Type 2 history - uses ALTER TABLE + UPDATE instead of overwrite
#
# TRANSFORMATION:
#   System_Role "SR-1" → SystemRoleID 1
#   System_Role "SR-10" → SystemRoleID 10
#
# STEPS:
#   1. Check if SystemRoleID column exists
#   2. If not, add SystemRoleID column (INT type) using ALTER TABLE
#   3. Extract numeric value using REGEXP_EXTRACT and UPDATE
#   4. Preserves all SCD Type 2 metadata (valid_from, valid_to, is_current)

print("Adding SystemRoleID column...")

# Check if column already exists
existing_columns = spark.table("thekitchen.s.moat_scoring").columns

if "SystemRoleID" not in existing_columns:
    # Step 1: Add column using ALTER TABLE (preserves all data)
    spark.sql("""
        ALTER TABLE thekitchen.s.moat_scoring 
        ADD COLUMN SystemRoleID INT
    """)
    print("✓ SystemRoleID column added to table schema")
else:
    print("ℹ️  SystemRoleID column already exists")

# Step 2: Update all records with extracted numeric value
# Uses REGEXP_EXTRACT to get digits after "SR-"
# Update with correct pattern - System_Role format is "SR-X Description"
# Use single backslash in triple-quoted string
update_result = spark.sql(r"""
    UPDATE thekitchen.s.moat_scoring
    SET SystemRoleID = TRY_CAST(REGEXP_EXTRACT(System_Role, 'SR-(\d+)', 1) AS INT)
    WHERE System_Role IS NOT NULL
""")

print(f"✓ Updated {update_result.collect()[0][0]} records")

print("✓ SystemRoleID values populated from System_Role")
print("  Extraction pattern: SR-X → X (as INT)")
print("  ✓ SCD Type 2 history preserved (no data loss)")

# Verify: Show sample of System_Role and new SystemRoleID
print("\nSample data:")
display(spark.table("thekitchen.s.moat_scoring").select("Company", "Date", "System_Role", "SystemRoleID", "is_current").orderBy("Company", "valid_from").limit(15))